In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import numpy as np
import pandas as pd
from IPython.display import display

print("Loading YAMNet model and class labels...")
model = hub.load('https://tfhub.dev/google/yamnet/1')

class_map_path = tf.keras.utils.get_file(
    'class_map.csv',
    'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
)
class_df = pd.read_csv(class_map_path)
class_names = class_df['display_name'].tolist()

# Define your 5 dataset folders and their mapping to YAMNet expected keywords
base_dir = '/content/drive/MyDrive/SafeCue Audio Recordings'
categories = {
    'Fire Alarm': ['fire alarm', 'smoke detector', 'alarm'],
    'Smoke Alarm': ['smoke detector', 'fire alarm', 'alarm'],
    'Siren': ['siren', 'ambulance (siren)', 'fire engine, emergency vehicle', 'police car (siren)'],
    'Screaming': ['shout', 'scream', 'yell'],
    'Background Noise': ['inside', 'outside', 'ambient', 'traffic', 'wind', 'chatter'] # Updated folder name here
}

all_results = []
CONFIDENCE_THRESHOLD = 0.35 # Adjust based on how strict you want the filter to be

for folder_name, expected_keywords in categories.items():
    folder_path = os.path.join(base_dir, folder_name)

    if not os.path.exists(folder_path):
        print(f"Directory missing: {folder_path}")
        continue

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.wav', '.mp3', '.flac'))]
    print(f"Processing {len(files)} files in '{folder_name}'...")

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        status = "KEEP"
        predicted_label = "Unknown"
        top_score = 0.0

        try:
            # Load audio at 16kHz mono as required by YAMNet
            wav, sr = librosa.load(file_path, sr=16000, mono=True)
            scores, embeddings, spectrogram = model(wav)
            mean_scores = tf.reduce_mean(scores, axis=0)
            top_class_index = tf.argmax(mean_scores).numpy()
            top_score = float(tf.reduce_max(mean_scores).numpy())

            predicted_label = class_names[top_class_index].lower()

            # Check if YAMNet prediction matches any expected keyword for this class
            # (Special handling for Background class where almost anything ambient is valid)
            if folder_name == 'Background':
                status = "KEEP" if top_score >= 0.15 else "FLAG_LOW_CONFIDENCE"
            else:
                match_found = any(keyword in predicted_label for keyword in expected_keywords)
                if not match_found or top_score < CONFIDENCE_THRESHOLD:
                    status = "FLAG_MISMATCH_OR_LOW"

        except Exception as e:
            status = f"ERROR: {str(e)}"

        all_results.append({
            "Category": folder_name,
            "File Name": file_name,
            "YAMNet Prediction": f"{predicted_label} ({top_score:.2f})",
            "Status": status
        })

# Convert to DataFrame and save audit log
audit_df = pd.DataFrame(all_results)
csv_output_path = os.path.join(base_dir, 'yamnet_audit_report.csv')
audit_df.to_csv(csv_output_path, index=False)

print(f"\nAudit complete! Report saved to {csv_output_path}")
display(audit_df[audit_df['Status'].str.startswith('FLAG')].head(15))

Loading YAMNet model and class labels...
Processing 207 files in 'Fire Alarm'...
Processing 196 files in 'Smoke Alarm'...
Processing 232 files in 'Siren'...
Processing 178 files in 'Screaming'...
Processing 265 files in 'Background Noise'...

Audit complete! Report saved to /content/drive/MyDrive/SafeCue Audio Recordings/yamnet_audit_report.csv


,Category,File Name,YAMNet Prediction,Status
4,Fire Alarm,kOfRBAucXV8_240.000_250.000.wav,speech (0.50),FLAG_MISMATCH_OR_LOW
5,Fire Alarm,Ni79i-5PfA4_500.000_510.000.wav,"smoke detector, smoke alarm (0.24)",FLAG_MISMATCH_OR_LOW
8,Fire Alarm,lTK9IgErMA8_50.000_60.000.wav,silence (0.46),FLAG_MISMATCH_OR_LOW
9,Fire Alarm,3kKfwoe57Eg_120.000_130.000.wav,speech (0.85),FLAG_MISMATCH_OR_LOW
10,Fire Alarm,MaMfDJzAsfc_90.000_100.000.wav,typing (0.20),FLAG_MISMATCH_OR_LOW
13,Fire Alarm,jB1xSB8MojQ_30.000_40.000.wav,"inside, small room (0.17)",FLAG_MISMATCH_OR_LOW
20,Fire Alarm,l_vWAJf-mvw_70.000_80.000.wav,speech (0.25),FLAG_MISMATCH_OR_LOW
21,Fire Alarm,icPBKqFUo_A_220.000_230.000.wav,alarm (0.35),FLAG_MISMATCH_OR_LOW
28,Fire Alarm,ioWizng27ok_90.000_100.000.wav,speech (0.41),FLAG_MISMATCH_OR_LOW
29,Fire Alarm,H1gsd1JV730_10.000_20.000.wav,speech (0.48),FLAG_MISMATCH_OR_LOW


In [ ]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import numpy as np
import pandas as pd
from IPython.display import display

print("Loading YAMNet model and class labels...")
model = hub.load('https://tfhub.dev/google/yamnet/1')

class_map_path = tf.keras.utils.get_file(
    'class_map.csv',
    'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
)
class_df = pd.read_csv(class_map_path)
class_names = class_df['display_name'].tolist()

BASE_DIR = '/content/drive/MyDrive/SafeCue Audio Recordings'
CLASSES = {
    'Fire Alarm': ['fire alarm', 'smoke detector', 'alarm'],
    'Smoke Alarm': ['smoke detector', 'fire alarm', 'alarm'],
    'Siren': ['siren', 'ambulance (siren)', 'fire engine, emergency vehicle', 'police car (siren)'],
    'Screaming': ['shout', 'scream', 'yell'],
    'Background Noise': ['inside', 'outside', 'ambient', 'traffic', 'wind', 'chatter']
}

all_evaluated_files = []

for folder_name, expected_keywords in CLASSES.items():
    folder_path = os.path.join(BASE_DIR, folder_name)

    if not os.path.exists(folder_path):
        print(f"Directory missing: {folder_path}")
        continue

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.wav', '.mp3', '.flac'))]
    print(f"Evaluating {len(files)} files in '{folder_name}'...")

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        top_score = 0.0
        predicted_label = "Unknown"

        try:
            wav, sr = librosa.load(file_path, sr=16000, mono=True)
            scores, embeddings, spectrogram = model(wav)
            mean_scores = tf.reduce_mean(scores, axis=0)
            top_class_index = tf.argmax(mean_scores).numpy()
            top_score = float(tf.reduce_max(mean_scores).numpy())
            predicted_label = class_names[top_class_index].lower()

            # For non-background classes, check if prediction aligns with expected keywords
            if folder_name != 'Background Noise':
                match_found = any(keyword in predicted_label for keyword in expected_keywords)
                # If it doesn't match the keyword at all, heavily penalize score so it drops to bottom
                if not match_found:
                    top_score = top_score * 0.1

        except Exception as e:
            top_score = -1.0 # Push errored files to the bottom
            predicted_label = f"Error: {str(e)}"

        all_evaluated_files.append({
            "Category": folder_name,
            "File Name": file_name,
            "File Path": file_path,
            "YAMNet Prediction": f"{predicted_label} ({top_score:.2f})",
            "Confidence Score": top_score
        })

# Convert to DataFrame
df_eval = pd.DataFrame(all_evaluated_files)

# Group by category, sort by confidence score descending, and take top 50 per folder
top_50_df = df_eval.groupby('Category', group_keys=False).apply(
    lambda x: x.sort_values(by='Confidence Score', ascending=False).head(50)
).reset_index(drop=True)

# Save summary report
output_csv = os.path.join(BASE_DIR, 'top_50_confident_clips.csv')
top_50_df.to_csv(output_csv, index=False)

print(f"\nTop 50 selection complete! Saved summary to {output_csv}")
display(top_50_df.groupby('Category')['File Name'].count())

Loading YAMNet model and class labels...
Evaluating 207 files in 'Fire Alarm'...
Evaluating 196 files in 'Smoke Alarm'...
Evaluating 232 files in 'Siren'...
Evaluating 178 files in 'Screaming'...
Evaluating 265 files in 'Background Noise'...

Top 50 selection complete! Saved summary to /content/drive/MyDrive/SafeCue Audio Recordings/top_50_confident_clips.csv


/tmp/ipykernel_1435/4241831405.py:76: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_50_df = df_eval.groupby('Category', group_keys=False).apply(


,File Name
Category,
Background Noise,50
Fire Alarm,50
Screaming,50
Siren,50
Smoke Alarm,50


In [ ]:
import os
import shutil
import pandas as pd

# Load the top 50 summary CSV created by the previous step
BASE_DIR = '/content/drive/MyDrive/SafeCue Audio Recordings'
csv_input_path = os.path.join(BASE_DIR, 'top_50_confident_clips.csv')

if not os.path.exists(csv_input_path):
    raise FileNotFoundError(f"Could not find {csv_input_path}. Run the scoring script first!")

top_50_df = pd.read_csv(csv_input_path)

print("Copying top 50 confident files into sibling folders...")

# Loop through each row in our top 50 dataframe
for _, row in top_50_df.iterrows():
    category = row['Category']
    file_name = row['File Name']
    src_path = row['File Path']

    # Define sibling folder path (e.g., "Fire Alarm_top50")
    sibling_folder_name = f"{category}_top50"
    sibling_folder_path = os.path.join(BASE_DIR, sibling_folder_name)

    # Create the sibling folder if it doesn't exist yet
    os.makedirs(sibling_folder_path, exist_ok=True)

    dst_path = os.path.join(sibling_folder_path, file_name)

    # Copy the file over if it exists in the source location
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
    else:
        print(f"Warning: Source file missing for copy -> {src_path}")

print("\n✓ All top 50 files successfully copied into their respective sibling folders!")

Copying top 50 confident files into sibling folders...

✓ All top 50 files successfully copied into their respective sibling folders!
